In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

**Learn the Basics** \|\| [Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Learn the Basics
================

Authors: [Suraj Subramanian](https://github.com/subramen), [Seth
Juarez](https://github.com/sethjuarez/), [Cassie
Breviu](https://github.com/cassiebreviu/), [Dmitry
Soshnikov](https://soshnikov.com/), [Ari
Bornstein](https://github.com/aribornstein/)

Most machine learning workflows involve working with data, creating
models, optimizing model parameters, and saving the trained models. This
tutorial introduces you to a complete ML workflow implemented in
PyTorch, with links to learn more about each of these concepts.

We\'ll use the FashionMNIST dataset to train a neural network that
predicts if an input image belongs to one of the following classes:
T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker,
Bag, or Ankle boot.

[This tutorial assumes a basic familiarity with Python and Deep Learning
concepts.]{.title-ref}

Running the Tutorial Code
-------------------------

You can run this tutorial in a couple of ways:

-   **In the cloud**: This is the easiest way to get started! Each
    section has a \"Run in Google Colab\" link at the top, which opens
    an integrated notebook in Google Colab with the code in a
    fully-hosted environment.
-   **Locally**: This option requires you to set up PyTorch and
    TorchVision first on your local machine ([installation
    instructions](https://pytorch.org/get-started/locally/)). Download
    the notebook or copy the code into your favorite IDE.

How to Use this Guide
---------------------

If you\'re familiar with other deep learning frameworks, check out the
[0. Quickstart](quickstart_tutorial.html) first to quickly familiarize
yourself with PyTorch\'s API.

If you\'re new to deep learning frameworks, head right into the first
section of our step-by-step guide: [1. Tensors](tensorqs_tutorial.html).

::: {.toctree maxdepth="2" hidden=""}
quickstart\_tutorial tensorqs\_tutorial data\_tutorial
transforms\_tutorial buildmodel\_tutorial autogradqs\_tutorial
optimization\_tutorial saveloadrun\_tutorial
:::


PyTorch 提供了两个用于处理数据的基本方法： torch.utils.data.DataLoader和torch.utils.data.Dataset。 Dataset存储样本及其对应的标签，并将DataLoader一个可迭代对象包装在Dataset。

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

PyTorch 提供了一些特定领域的库，例如TorchText、 TorchVision和TorchAudio，它们都包含数据集。在本教程中，我们将使用 TorchVision 数据集。

该torchvision.datasets模块包含Dataset适用于多种真实世界视觉数据集的对象，例如 CIFAR 和 COCO（完整列表请点击此处）。在本教程中，我们将使用 FashionMNIST 数据集。每个 TorchVision 对象Dataset都包含两个参数： `samples` 和 `labels` transform，分别 target_transform用于修改样本和标签。

In [3]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 17.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 269kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.06MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 32.4MB/s]


我们将数据加载器Dataset作为参数传递给它DataLoader。这会封装一个包含我们数据集的可迭代对象，并支持自动批处理、采样、打乱顺序和多进程数据加载。这里我们定义批处理大小为 64，即数据加载器可迭代对象中的每个元素都会返回一个包含 64 个特征和标签的批次。

In [4]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


创建模型
要在 PyTorch 中定义神经网络，我们创建一个继承自`nn.Module`的类。我们在函数中定义网络的层__init__，并在函数中指定数据在网络中的传递方式forward。为了加速神经网络的运算，我们将其迁移到加速器，例如 CUDA、MPS、MTIA 或 XPU。如果当前有可用的加速器，我们将使用它；否则，我们将使用 CPU。

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


要训​​练一个模型，我们需要一个损失函数 和一个优化器。

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

在一个训练循环中，模型对训练数据集（分批输入）进行预测，并将预测误差反向传播以调整模型的参数。

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

我们还会针对测试数据集检查模型的性能，以确保它能够学习。

In [8]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

训练过程需要经过多次迭代（epoch）。在每个epoch中，模型都会学习参数以做出更准确的预测。我们会记录每个epoch的模型准确率和损失值；我们希望看到准确率随着epoch的增加而提高，损失值随着epoch的增加而降低。

In [9]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.317677  [   64/60000]
loss: 2.306569  [ 6464/60000]
loss: 2.287211  [12864/60000]
loss: 2.274234  [19264/60000]
loss: 2.268520  [25664/60000]
loss: 2.238730  [32064/60000]
loss: 2.243142  [38464/60000]
loss: 2.219152  [44864/60000]
loss: 2.221510  [51264/60000]
loss: 2.180600  [57664/60000]
Test Error: 
 Accuracy: 41.1%, Avg loss: 2.183402 

Epoch 2
-------------------------------
loss: 2.203076  [   64/60000]
loss: 2.193412  [ 6464/60000]
loss: 2.141864  [12864/60000]
loss: 2.146863  [19264/60000]
loss: 2.110762  [25664/60000]
loss: 2.055756  [32064/60000]
loss: 2.077851  [38464/60000]
loss: 2.019448  [44864/60000]
loss: 2.027776  [51264/60000]
loss: 1.938393  [57664/60000]
Test Error: 
 Accuracy: 58.8%, Avg loss: 1.951752 

Epoch 3
-------------------------------
loss: 1.990896  [   64/60000]
loss: 1.963425  [ 6464/60000]
loss: 1.855253  [12864/60000]
loss: 1.877916  [19264/60000]
loss: 1.779705  [25664/60000]
loss: 1.723126  [32064/600

节省模型
保存模型的一种常见方法是序列化内部状态字典（包含模型参数）。

In [10]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


正在加载模型
加载模型的过程包括重新创建模型结构并将状态字典加载到其中。

In [11]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

现在可以利用该模型进行预测。

In [12]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


# 张量

Tensors are a specialized data structure that are very similar to arrays and matrices. In PyTorch, we use tensors to encode the inputs and outputs of a model, as well as the model’s parameters.
张量是一种与数组和矩阵非常相似的专用数据结构。在 PyTorch 中，我们使用张量编码模型的输入和输出，以及模型参数。

Tensors are similar to NumPy’s ndarrays, except that tensors can run on GPUs or other hardware accelerators. In fact, tensors and NumPy arrays can often share the same underlying memory, eliminating the need to copy data (see Bridge with NumPy). Tensors are also optimized for automatic differentiation (we’ll see more about that later in the Autograd section). If you’re familiar with ndarrays, you’ll be right at home with the Tensor API. If not, follow along!
张量类似于 NumPy 的 ndarray，不同之处在于张量可以在 GPU 或其他硬件加速器上运行。事实上，张量和 NumPy 数组通常可以共享相同的底层内存，从而无需复制数据（参见 NumPy 桥接 ）。张量也针对自动微分进行了优化（我们稍后会在自动分列中详细介绍 部分）。如果你熟悉 ndarrays，Tensor API 你会非常熟悉。如果没有，就跟着看吧！

In [13]:
import torch
import numpy as np

Initializing a Tensor  张量初始化#
Tensors can be initialized in various ways. Take a look at the following examples:
张量可以通过多种方式初始化。请看以下示例：

Directly from data  直接数据

Tensors can be created directly from data. The data type is automatically inferred.
张量可以直接从数据中生成。数据类型会自动推断。


In [14]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

From a NumPy array  来自 NumPy 数组

Tensors can be created from NumPy arrays (and vice versa - see Bridge with NumPy).
张量可以由 NumPy 数组创建（反之亦然——参见 NumPy 桥接 ）。

In [15]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

From another tensor:  另一个张量：

The new tensor retains the properties (shape, datatype) of the argument tensor, unless explicitly overridden.
新张量保留参数张量的性质（形状、数据类型），除非被显式覆盖。

In [16]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.2673, 0.7699],
        [0.3319, 0.1551]]) 



With random or constant values:
随机或恒定值：

shape is a tuple of tensor dimensions. In the functions below, it determines the dimensionality of the output tensor.
形状是张量维数的元组。在下面的函数中，它决定了输出张量的维数。

In [17]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.6758, 0.3463, 0.2115],
        [0.2192, 0.2331, 0.2522]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


Attributes of a Tensor  张量的属性#
Tensor attributes describe their shape, datatype, and the device on which they are stored.
张量属性描述了它们的形状、数据类型以及存储设备的特性。

In [18]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


Operations on Tensors  张量运算#
Over 1200 tensor operations, including arithmetic, linear algebra, matrix manipulation (transposing, indexing, slicing), sampling and more are comprehensively described here.
这里全面描述了 1200 多种张量运算，包括算术、线性代数、矩阵作（换位、指标、切片）、采样等。

Each of these operations can be run on the CPU and Accelerator such as CUDA, MPS, MTIA, or XPU. If you’re using Colab, allocate an accelerator by going to Runtime > Change runtime type > GPU.
这些作都可以在 CPU 和加速器上运行 例如 CUDA、MPS、MTIA 或 XPU。如果你用的是 Colab，可以选择 Runtime > Change runtime type > GPU 来分配加速器。

By default, tensors are created on the CPU. We need to explicitly move tensors to the accelerator using .to method (after checking for accelerator availability). Keep in mind that copying large tensors across devices can be expensive in terms of time and memory!
默认情况下，张量是在 CPU 上创建的。我们需要显式地将张量移动到加速器，使用 .to 方法（在检查加速器的可用性后）。请记住，跨设备复制大型张量可能会花费大量时间和内存！

In [19]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

Try out some of the operations from the list. If you’re familiar with the NumPy API, you’ll find the Tensor API a breeze to use.
试试清单上的一些作。如果你熟悉 NumPy API，会发现 Tensor API 使用起来非常简单。

Standard numpy-like indexing and slicing:
标准的类数字索引和切片：

In [28]:
tensor = torch.rand(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[:,-1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([0.0382, 0.0441, 0.8146, 0.4242])
First column: tensor([0.0382, 0.6564, 0.7394, 0.4059])
Last column: tensor([0.4242, 0.2695, 0.0550, 0.3673])
tensor([[0.0382, 0.0000, 0.8146, 0.4242],
        [0.6564, 0.0000, 0.2618, 0.2695],
        [0.7394, 0.0000, 0.5852, 0.0550],
        [0.4059, 0.0000, 0.2057, 0.3673]])


Joining tensors You can use torch.cat to concatenate a sequence of tensors along a given dimension. See also torch.stack, another tensor joining operator that is subtly different from torch.cat.
连接张量你可以用 torch.cat 在给定维度上串接一系列张量。另见 torch.stack，另一种与 torch.cat 有细微不同但又不同的张量连接算子。

In [29]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[0.0382, 0.0000, 0.8146, 0.4242, 0.0382, 0.0000, 0.8146, 0.4242, 0.0382,
         0.0000, 0.8146, 0.4242],
        [0.6564, 0.0000, 0.2618, 0.2695, 0.6564, 0.0000, 0.2618, 0.2695, 0.6564,
         0.0000, 0.2618, 0.2695],
        [0.7394, 0.0000, 0.5852, 0.0550, 0.7394, 0.0000, 0.5852, 0.0550, 0.7394,
         0.0000, 0.5852, 0.0550],
        [0.4059, 0.0000, 0.2057, 0.3673, 0.4059, 0.0000, 0.2057, 0.3673, 0.4059,
         0.0000, 0.2057, 0.3673]])


Arithmetic operations  算术运算

In [30]:
# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T#矩阵乘法
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)


# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor#逐个元素相乘
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor([[0.0015, 0.0000, 0.6635, 0.1800],
        [0.4309, 0.0000, 0.0686, 0.0726],
        [0.5468, 0.0000, 0.3425, 0.0030],
        [0.1647, 0.0000, 0.0423, 0.1349]])

Single-element tensors If you have a one-element tensor, for example by aggregating all values of a tensor into one value, you can convert it to a Python numerical value using item():
单元素张量例如，如果你有一个单元素张量，比如将一个张量的所有值聚合为一个值，你可以用 item（） 将其转换为 Python 的数值

In [31]:
agg = tensor.sum() #对张量tensor中的所有元素求和，返回一个零维张量（标量张量），形状为()，包含一个值。
print(agg,type(agg))
agg_item = agg.item() #将这个标量张量转换为Python数值（如float或int）
print(agg_item, type(agg_item))

tensor(4.8232) <class 'torch.Tensor'>
4.823184967041016 <class 'float'>
